## 라이브러리 호출 및 데이터 로드

In [1]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json
from pprint import pprint
import warnings
import platform

# 통계용 라이브러리 호출
from scipy import stats
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from itertools import combinations
from matplotlib.patches import Patch
import pingouin as pg
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize, proportion_confint
from statsmodels.stats.power import NormalIndPower

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

In [2]:
# 데이터 로드
df_champion = pd.read_csv('../../유저단위_게임데이터_상위랭커보존-stats_champion_1.csv')
df_combination = pd.read_csv('../../유저단위_게임데이터_상위랭커보존-stats_combination_1.csv')
df_champion_with_items = pd.read_csv("../../유저단위_게임데이터_상위랭커보존-stats_champion_items_1.csv")

---
### **경기데이터와 챔피언 데이터가 결합된 테이블**
- `df_champion`

In [5]:
df_champion.info()

<class 'pandas.DataFrame'>
RangeIndex: 396204 entries, 0 to 396203
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   user_id           396204 non-null  str  
 1   game_id           396204 non-null  str  
 2   user_tier         396204 non-null  str  
 3   ranked            396204 non-null  int64
 4   flag_1            396204 non-null  int64
 5   flag_2            396204 non-null  int64
 6   active_synergies  396204 non-null  str  
 7   top4_flag         396204 non-null  bool 
 8   ranked_1          396204 non-null  bool 
 9   champions         396204 non-null  str  
dtypes: bool(2), int64(3), str(5)
memory usage: 24.9 MB


In [6]:
df_champion.head(1)

,user_id,game_id,user_tier,ranked,flag_1,flag_2,active_synergies,top4_flag,ranked_1,champions
0,KR-USER-1,KR_4291707834,platinum,5,0,0,{},False,False,"[{'name': 'ziggs', 'star': 1, 'cost': 1, 'origin': 'Rebel', 'class': ""['Demolitionist']""}, {'name': 'ashe', 'star': 1, 'cost': 3, 'origin': 'Celestial', 'class': ""['Sniper']""}, {'name': 'chogath', 'star': 1, 'cost': 4, 'origin': 'Void', 'class': ""['Brawler']""}, {'name': 'ekko', 'star': 1, 'cost': 5, 'origin': 'Cybernetic', 'class': ""['Infiltrator']""}]"


#### 🔻분석단위를 챔피언 단위로 변경
- 하나의 row = 한 유저가 참여한 하나의 게임
- 챔피언마다 name, star, cost, origin, class 정보가 담겨있음 

In [7]:
# champions 컬럼 문자열 → 리스트로 변환
df_champion['champions'] = df_champion['champions'].apply(ast.literal_eval)

# 챔피언 단위로 분리 (explode)
df_champ_exploded = df_champion.explode('champions')

# 딕셔너리 컬럼 분리
df_champ_exploded = pd.concat([
    df_champ_exploded.drop(columns='champions'),
    df_champ_exploded['champions'].apply(pd.Series)
], axis=1)
# champion 컬럼 분리 전
# champions = {'name': 'ziggs', 'star': 1, 'cost': 1, 'origin': 'Rebel', 'class': ['Demolitionist']}

# champion 컬럼 분리 후
# user_id    game_id  ranked  top4_flag   name   star  cost  origin         class
# KR-USER-1  KR_...     5       False     ziggs   1     1    Rebel     [Demolitionist]
# KR-USER-1  KR_...     5       False     ashe    1     3   Celestial     [Sniper]

print(df_champ_exploded.shape)
display(df_champ_exploded.head(1))

(3130701, 14)


,user_id,game_id,user_tier,ranked,flag_1,flag_2,active_synergies,top4_flag,ranked_1,name,star,cost,origin,class
0,KR-USER-1,KR_4291707834,platinum,5,0,0,{},False,False,ziggs,1,1,Rebel,['Demolitionist']


In [8]:
# flag_1 제거 (챔피언 정보 누락된 row 제거)
df_champ_exploded = df_champ_exploded[df_champ_exploded['flag_1'] == 0]

print(df_champ_exploded.shape)

(3130701, 14)


---
### **경기데이터와 콤비네이션 데이터가 결합된 테이블**
- `df_combination`

---
### **경기데이터와 챔피언 데이터, 아이템 데이터가 결합된 테이블**
- `df_champion_with_items`